## Listening Task

In [ ]:
# Imports used in this notebook
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [ ]:
# Listening task data import
df = pd.read_csv('ListeningTaskFinal.csv')
df

In [ ]:
# Check to see which columns are included in the df for deletion of irrelevant columns
df.columns

In [ ]:
# Remove those columns
columns_to_be_removed = [ "StartDate", "EndDate", "Status", "IPAddress", "Progress", "Duration (in seconds)", "Finished",
    "RecordedDate", "ResponseId", "RecipientFirstName", "RecipientLastName", "RecipientEmail", "ExternalReference", "LocationLatitude",
    "LocationLongitude", "DistributionChannel", "UserLanguage"]

df_columns_removed = df.drop(columns=columns_to_be_removed)
df_columns_removed

In [ ]:
# Remove explanation rows, and an accidental double entry from participant (by hand)
df_columns_removed = df_columns_removed.drop(index=[0, 1, 6]).reset_index(drop=True)
df_columns_removed

In [ ]:
# Columns containing personalness ratings for each recording
columns_for_personalness = ["1_1", "2_1", "3_1", "4_1", "5_1"]

# Columns containing meaningfulness ratings for each recording
columns_for_meaningfulness = ["6_1", "7_1", "8_1", "9_1", "10_1"]

# Columns containing supportiveness for coping with homesickness ratings for each recording
columns_for_supportiveness = ["11_1", "12_1", "13_1", "14_1", "15_1"]

# S Map all text values of the ratings to scores
personalness_cell_values = { "Very generic": 1, "Generic": 2, "Somewhat generic": 3, "Neutral": 4, 
                         "Somewhat personal": 5, "Personal": 6,"Very personal": 7}
meaningfulness_cell_values = {"Not meaningful at all": 1, "Barely meaningful": 2, "Slightly meaningful": 3,
    "Neutral": 4, "Somewhat meaningful": 5, "Meaningful": 6, "Very meaningful": 7}
supportiveness_cell_values = {"Not supportive at all": 1, "Barely supportive": 2, "Slightly supportive": 3,
    "Neutral": 4, "Somewhat supportive": 5, "Supportive": 6, "Very supportive": 7}

# Replace the text ratings with the number scores
df_columns_removed[columns_for_personalness] = ( df_columns_removed[columns_for_personalness].replace(personalness_cell_values).astype(float))
df_columns_removed[columns_for_meaningfulness] = ( df_columns_removed[columns_for_meaningfulness].replace(meaningfulness_cell_values).astype(float))
df_columns_removed[columns_for_supportiveness] = ( df_columns_removed[columns_for_supportiveness].replace(supportiveness_cell_values).astype(float))

# Check if its converted correctly
df_columns_removed

In [ ]:
# Columns have vague names now, so rename them for clarity
score_columns_new_names = {"1_1": "Rec 1 - Personalness", "6_1": "Rec 1 - Meaningfulness", "11_1": "Rec 1 - Supportiveness",
                           "2_1": "Rec 2 - Personalness", "7_1": "Rec 2 - Meaningfulness", "12_1": "Rec 2 - Supportiveness",
                           "3_1": "Rec 3 - Personalness", "8_1": "Rec 3 - Meaningfulness", "13_1": "Rec 3 - Supportiveness",
                           "4_1": "Rec 4 - Personalness", "9_1": "Rec 4 - Meaningfulness", "14_1": "Rec 4 - Supportiveness",
                           "5_1": "Rec 5 - Personalness", "10_1": "Rec 5 - Meaningfulness", "15_1": "Rec 5 - Supportiveness"}

df_columns_removed = df_columns_removed.rename(columns=score_columns_new_names)
df_columns_removed

In [ ]:
# New df containing averages of scores for each recording, for each variable
average_of_recordings = pd.DataFrame({
    "Recording": [1, 2, 3, 4, 5], "Personalness Mean": [ df_columns_removed["Rec 1 - Personalness"].mean(),
                                                         df_columns_removed["Rec 2 - Personalness"].mean(),
                                                         df_columns_removed["Rec 3 - Personalness"].mean(),
                                                         df_columns_removed["Rec 4 - Personalness"].mean(),
                                                         df_columns_removed["Rec 5 - Personalness"].mean()],
                                "Meaningfulness Mean": [ df_columns_removed["Rec 1 - Meaningfulness"].mean(),
                                                         df_columns_removed["Rec 2 - Meaningfulness"].mean(),
                                                         df_columns_removed["Rec 3 - Meaningfulness"].mean(),
                                                         df_columns_removed["Rec 4 - Meaningfulness"].mean(),
                                                         df_columns_removed["Rec 5 - Meaningfulness"].mean()],
                                "Supportiveness Mean": [ df_columns_removed["Rec 1 - Supportiveness"].mean(),
                                                         df_columns_removed["Rec 2 - Supportiveness"].mean(),
                                                         df_columns_removed["Rec 3 - Supportiveness"].mean(),
                                                         df_columns_removed["Rec 4 - Supportiveness"].mean(),
                                                         df_columns_removed["Rec 5 - Supportiveness"].mean()]})

average_of_recordings

In [ ]:
## Calculating the highest scores for each recording, for each variable
# Columns of the variables
variables = ["Personalness Mean", "Meaningfulness Mean", "Supportiveness Mean"]

# Calculate the highest value for each variable and find which recording belong to that value
for variable in variables:
    #highest value of each variable
    highest_value = average_of_recordings[variable].max()
    # In the df, for each highest value, see which recording corresponds to it
    highest_recordings = average_of_recordings.loc[average_of_recordings[variable] == highest_value, ["Recording", variable]]
    # Shows the recordings with highest values and the value itself. The first numbers that are seen, 2, 3, 2, are indexes and should be ignored 
    print(highest_recordings)

# Calculate the lowest value for each variable and find which recording belong to that value
for variable in variables:
    #lowest value of each variable
    lowest_value = average_of_recordings[variable].min()
    # In the df, for each lowest value, see which recording corresponds to it
    lowest_recordings = average_of_recordings.loc[average_of_recordings[variable] == lowest_value, ["Recording", variable]]
    # Shows the recordings with lowest values and the value itself. The first numbers that are seen, 0, 0, 0, are indexes and should be ignored 
    print(lowest_recordings)

In [ ]:
## Calculating the mean, standard deviation, and variance for each recording, for each variable

# All df columns
df_columns_removed_columns = [ "Rec 1 - Personalness", "Rec 2 - Personalness", "Rec 3 - Personalness", "Rec 4 - Personalness", "Rec 5 - Personalness",
                               "Rec 1 - Meaningfulness", "Rec 2 - Meaningfulness", "Rec 3 - Meaningfulness", "Rec 4 - Meaningfulness", "Rec 5 - Meaningfulness",
                               "Rec 1 - Supportiveness", "Rec 2 - Supportiveness", "Rec 3 - Supportiveness", "Rec 4 - Supportiveness", "Rec 5 - Supportiveness"]

# All df variables
variables = ["Personalness", "Meaningfulness", "Supportiveness"]

# Empty row list
rows = []

# For each row and column in the df:
for i, col in enumerate(df_columns_removed_columns):
    # Take the variable needed
    variable = variables[i // 5]
    # Take the recording number needed
    recording = (i % 5) + 1

    # Get the given scores for each recording, for each participant
    values = pd.to_numeric(df_columns_removed[col])

    # Add the recording number, its mean rating score, its SD, and its variance to the empty rows list
    rows.append({"Recording": recording,
                 "Variable": variable,
                 "Mean": values.mean(),
                 "SD": values.std(ddof=1),
                 "Variance": values.var(ddof=1)})

# New df containing the above information using the rows list
df_listening_calculations = pd.DataFrame(rows)

# Round the mean, SD, and variance on two decimals
df_listening_calculations = df_listening_calculations.round({ "Mean": 2, "SD": 2, "Variance": 2})
df_listening_calculations

### Next Part

## Initial Survey

In [ ]:
# Load initial survey into df2
df2 = pd.read_csv('PreQuestionnaire.csv')
df2

In [ ]:
# Check all columns to see which irrelevant ones can be removed 
df2.columns

In [ ]:
# Remove those columns
columns_to_be_removed_df2 = ["StartDate", "EndDate", "Status", "IPAddress", "Duration (in seconds)",
                             "RecordedDate", "ResponseId", "RecipientFirstName", "RecipientLastName",
                             "RecipientEmail", "ExternalReference", "LocationLatitude", "LocationLongitude",
                             "DistributionChannel", "UserLanguage", "Intro", "Preference", "Finished",
                             "Progress", "Homesickness"]
df2_columns_removed = df2.drop(columns=columns_to_be_removed_df2)

# Rename Q2 to Age for clarity
df2_columns_removed = df2_columns_removed.rename(columns={"Q2": "Age"})
df2_columns_removed

In [ ]:
# Remove unnecessary rows (dont contain user data)
df2_columns_removed = df2_columns_removed.drop(index=[0, 1])
# Reset index
df2_columns_removed = df2_columns_removed.reset_index(drop=True)

# No entries in 'other' gender category, so remove column
# Variable for Gender_5_TEXT columns (contains potential other genders)
other_genders = df2_columns_removed["Gender_5_TEXT"]

# Make all NaN values the same if not already
other_genders_cleaned = (other_genders.astype("string").str.strip()
        .replace(["", "NaN", "nan", "NAN", "None", "none", "NULL", "null", "<NA>"], np.nan))

# If all values for that column are NaN, drop the column
if other_genders_cleaned.dropna().empty:
    df2_columns_removed = df2_columns_removed.drop(columns=["Gender_5_TEXT"])

# All columns containing to the UHS questions.
uhs_columns = ["Q12_1", "Q12_2", "Q12_3", "Q12_4", "Q12_5",
               "Q12_6", "Q12_7", "Q12_8", "Q12_9", "Q12_10",
               "Q12_11", "Q12_12", "Q12_13", "Q12_14", "Q12_15",
               "Q12_16", "Q12_17", "Q12_18", "Q12_19", "Q12_20"]

# Turn all NaN-type values into the same, actual NaN value instead of potential strings.
df2_columns_removed[uhs_columns] = df2_columns_removed[uhs_columns].replace(
    ["NaN", "nan", "NAN", ""],
    np.nan
)

# Drop rows where any Q12 column contains NaN
df2_full_entries = df2_columns_removed.dropna(subset=uhs_columns)

# Removing my own test entry
df2_full_entries = df2_full_entries.drop(index=0)

# Reset index
df2_full_entries = df2_full_entries.reset_index(drop=True)

# Removing any entries, by hand, of people that have filled in being from the same birth country as where they are currently located
df2_full_entries = df2_full_entries.drop(index=[4, 9, 33])
df2_full_entries = df2_full_entries.reset_index(drop=True)
df2_full_entries

## UHS Scores

In [ ]:
# All uhs columns
uhs_columns = ["Q12_1", "Q12_2", "Q12_3", "Q12_4", "Q12_5",
               "Q12_6", "Q12_7", "Q12_8", "Q12_9", "Q12_10",
               "Q12_11", "Q12_12", "Q12_13", "Q12_14", "Q12_15",
               "Q12_16", "Q12_17", "Q12_18", "Q12_19", "Q12_20"]

# Which words correspond to which UHS score given
uhs_numbers = {"not": 1, "weak": 2, "moderate": 3, "strong": 4, "very strong": 5}

# Change UHS words to numbers for each cell, so getting a UHS number score instead of text
df2_full_entries[uhs_columns] = (df2_full_entries[uhs_columns].astype("string").stack().str.strip().str.lower()
                                .replace(uhs_numbers).unstack().astype(float))

# Calculate the mean of each participant and add a column to the df
df2_full_entries["UHS avg"] = df2_full_entries[uhs_columns].mean(axis=1)
df2_full_entries

In [ ]:
# Calculate lest, highest, median, and average UHS scores given by participants
lowest_uhs = df2_full_entries["UHS avg"].min()
highest_uhs = df2_full_entries["UHS avg"].max()
med_uhs = df2_full_entries["UHS avg"].median()
avg_uhs = df2_full_entries["UHS avg"].mean().round(2)
print(lowest_uhs, highest_uhs, med_uhs, avg_uhs)

In [ ]:
# Make numeric if not already
df2_full_entries["Age"] = pd.to_numeric(df2_full_entries["Age"], errors="coerce")

# Descriptive of age
descriptives_age = df2_full_entries["Age"].describe()
descriptives_age

In [ ]:
# Age median
df2_full_entries["Age"].median()

In [ ]:
# Amount per gender, no other genders than male or female (checked earlier in the code)
amount_per_gender = df2_full_entries["Gender"].value_counts(dropna=False)
amount_per_gender

In [ ]:
# Amount of relocations per reason
reasons_for_relocation_count = df2_full_entries["Relocation Reason"].value_counts(dropna=False)
reasons_for_relocation_count

In [ ]:
# Make sure UHS avg is numeric
df2_full_entries["UHS avg"] = pd.to_numeric(df2_full_entries["UHS avg"], errors="coerce")

# Creative interpretive buckets for UHS scores
bins = [1.00, 1.50, 2.50, 3.50, 4.50, 5.01]

# Labels corresponding to the buckets
labels = ["Very low", "Low", "Moderate", "High", "Very high"]

# Add column containing the interpretive UHS text to the df
df2_full_entries["UHS descriptive"] = pd.cut(df2_full_entries["UHS avg"], bins=bins, labels=labels, right=False)
df2_full_entries

In [ ]:
df2_full_entries.columns

In [ ]:
## All individual UHS score columns (to be removed for thesis table)
remove_cols = [f"Q12_{i}" for i in range(1, 21)]

# Remove these for the thesis table
df2_thesis = df2_full_entries.drop(columns=remove_cols, errors="ignore")

# Increase index so it starts at 1
df2_thesis.index = df2_thesis.index + 1
df2_thesis

In [ ]:
# Minimum, maximum, mean, standard deviation, and median average UHS scores
uhs_avg_min = df2_thesis["UHS avg"].min()
uhs_avg_max = df2_thesis["UHS avg"].max()
uhs_avg_mean = df2_thesis["UHS avg"].mean()
uhs_avg_std = df2_thesis["UHS avg"].std()
uhs_avg_med = df2_thesis["UHS avg"].median()
print(uhs_avg_min, uhs_avg_max, uhs_avg_mean, uhs_avg_std, uhs_avg_med)

In [ ]:
# UHS average insights for each gender
gender_uhs_insights = df2_thesis.groupby("Gender")["UHS avg"].agg(count="count", mean="mean", sd="std",
                                                                  median="median", min="min", max="max").round(2)

gender_uhs_insights

In [ ]:
# UHS average insights for each relocation reason
relocation_uhs_insights = df2_thesis.groupby("Relocation Reason")["UHS avg"].agg(count="count",mean="mean",
                                              sd="std",median="median",min="min",max="max").round(2)

relocation_uhs_insights

In [ ]:
# df_columns_removed is the dataframe containing identifiers, and every score given by participants for every recording and every variable, in numbers
# df2_thesis contains those same identifiers, and a column containing the average UHS score for each participant
## Now I will merge them based on the identifier. Of course, there are participants who only participated in the first questionnaire and will be left out.
# Identifiers will be lowercased and spaces will be removed
# Also, any almost-matching identifiers will be matched by hand
# Then, the UHS average score column from df2_final will be added to df_columns_removed

# Rename 'Contact Information' to 'Identifier' in df2_thesis
df2_thesis = df2_thesis.rename(columns={"Contact Information": "identifier"})

# Copy just to be sure
df_initial_survey = df2_thesis[["identifier", "UHS avg"]].copy()

# Lowercase and remove any spacing in both initial survey and listening task questionnaire
df2_thesis["identifier"] = df2_thesis["identifier"].astype(str).str.strip().str.lower()
df_columns_removed["identifier"] = df_columns_removed["identifier"].astype(str).str.strip().str.lower()

# All unique identifiers in listening task df, removing all NaNs
listening_task_ids = df_columns_removed["identifier"].dropna().unique()

# In the initial survey df, keep the rows where identifiers match with those of the listening task
df2_thesis_listening_participants_only = df2_thesis[df2_thesis["identifier"].isin(listening_task_ids)].copy()

# Take the identifier and average UHS score column from the newly formed df
df_columns_removed_listening_participants_only = df2_thesis_listening_participants_only[["identifier", "UHS avg"]].copy()

# Keep only the rows of participants for which the identifier is in the listening task df and the initial survey df
df_columns_removed_listening_participants_only[df_columns_removed_listening_participants_only["identifier"].duplicated(keep=False)].sort_values("identifier")

# Merge the dfs based on the identifiers
df_correlation = df_columns_removed.merge(df_columns_removed_listening_participants_only, on="identifier", how="left", validate="one_to_one")
df_correlation

In [ ]:
# Check where NaN values exist in the UHS avg column
df_correlation[df_correlation["UHS avg"].isna()]

In [ ]:
# The correlation dataframe now shows 3 NaN values, because the identifiers of the two merged dataframes aren't the same, 
# so I will manually check df2_final for the UHS avg scores and add them manually:
df2_thesis

#### Code below had identifiers manually put in, because of privacy, I removed these and wrote 'phone number' or 'email address'

In [ ]:
# Manually add missing UHS avg scores to the correlation dataframe
df_correlation.loc[df_correlation["identifier"] == "+31 619021708", "UHS avg"] = 3.55
df_correlation.loc[df_correlation["identifier"] == "francesorelly@att.net", "UHS avg"] = 3.00
df_correlation.loc[df_correlation["identifier"] == "connieaibhilin@gmail.com", "UHS avg"] = 3.40

In [ ]:
# Now all participants have their ratings of the different recordings and variables, as well as the UHS avg score, in one dataframe
df_correlation

In [ ]:
df2_thesis

In [ ]:
# Create copies just to be sure
df2_thesis_copy = df2_thesis.copy()
df_correlation_copy = df_correlation.copy()

# Clean accidental spaces in identifiers
df2_thesis_copy["identifier"] = df2_thesis_copy["identifier"].astype(str).str.strip()
df_correlation_copy["identifier"] = df_correlation_copy["identifier"].astype(str).str.strip()

# Fix identifiers that are wrong in df_correlation
# left = wrong value in df_correlation, right = correct value matching df2_thesis
correlation_identifier_fixes = {
    "francesorelly@att.net": "francesoreilly@att.net",
    "+31 619021708": "+31619021708",
}

# Replace the wrong ids with the correct ids in the correlation df
for wrong_id, correct_id in correlation_identifier_fixes.items():
    df_correlation_copy.loc[
        df_correlation_copy["identifier"] == wrong_id,
        "identifier"
    ] = correct_id

# Fix identifiers that are wrong in df2_thesis
# left = wrong value in df2_thesis, right = correct value matching df_correlation
df2_identifier_fixes = {
    "connieaibhilin@gmail.com / +31 639381988": "connieaibhilin@gmail.com",
}

for wrong_id, correct_id in df2_identifier_fixes.items():
    df2_thesis_copy.loc[
        df2_thesis_copy["identifier"] == wrong_id,
        "identifier"
    ] = correct_id
df_correlation_copy

In [ ]:
# All demographics columns
demographics_cols = ["identifier", "Age", "Gender", "Relocation Reason"]

# Create new df for showing all interview/listening task participants and their identifiers, UHS averages, ages, genders, and relocation reasons
# by merging on identifiers of the intial survey df and listening task df (these identifiers are now corrected in the cells above
# so they will now merge correctly)
interview_demographics_df = df_correlation_copy[["identifier", "UHS avg"]].merge(df2_thesis_copy[demographics_cols], on="identifier", how="left")

interview_demographics_df

In [ ]:
# Descriptives of age for interview/listening-task participants
descriptives_age_interview = interview_demographics_df["Age"].describe()
descriptives_age_interview

In [ ]:
# Median age for interview/listening-task participants
median_age_interview = interview_demographics_df["Age"].median()
median_age_interview

In [ ]:
# Amount per gender for interview/listening-task participants
amount_per_gender_interview = interview_demographics_df["Gender"].value_counts(dropna=False)
amount_per_gender_interview

In [ ]:
# Relocation reasons for interview/listening-task participants
reasons_for_relocation_count_interview = interview_demographics_df["Relocation Reason"].value_counts(dropna=False)
reasons_for_relocation_count_interview

In [ ]:
# UHS descriptives for interview/listening-task participants
uhs_descriptives_interview = interview_demographics_df["UHS avg"].describe().round(2)
uhs_descriptives_interview

In [ ]:
# UHS median for interview/listening-task participants
uhs_median_interview = interview_demographics_df["UHS avg"].median()
uhs_median_interview

## Think-Aloud

#### Code below had identifiers manually put in, because of privacy, I removed these and wrote 'phone number' or 'email address'

In [ ]:
# Think-aloud participant subgroup descriptives (n = 3)

# Fill in the identifiers of the three think-aloud participants by hand
identifiers_think_aloud = ["email address, phone number, email address"]

# Clean identifiers for correct matching
identifiers_think_aloud = [str(identifier).strip().lower() for identifier in identifiers_think_aloud]

# Create dataframe for think-aloud participants
df_think_aloud = interview_demographics_df[interview_demographics_df["identifier"].isin(identifiers_think_aloud)].copy()

# Remove identifier column, not needed
df_think_aloud = df_think_aloud.drop(columns=["identifier"])

# Check result
print("Number of think-aloud participants:", len(df_think_aloud))
df_think_aloud

In [ ]:
# Make sure Age and UHS avg are numeric
df_think_aloud["Age"] = pd.to_numeric(df_think_aloud["Age"], errors="coerce")
df_think_aloud["UHS avg"] = pd.to_numeric(df_think_aloud["UHS avg"], errors="coerce")

In [ ]:
# Descriptives of age for think-aloud participants
descriptives_age_think_aloud = df_think_aloud["Age"].describe()
descriptives_age_think_aloud

In [ ]:
# Median age for think-aloud participants
median_age_think_aloud = df_think_aloud["Age"].median()
median_age_think_aloud

In [ ]:
# Amount per gender for think-aloud participants
amount_per_gender_think_aloud = df_think_aloud["Gender"].value_counts(dropna=False)
amount_per_gender_think_aloud

In [ ]:
# Relocation reasons for think-aloud participants
reasons_for_relocation_count_think_aloud = df_think_aloud["Relocation Reason"].value_counts(dropna=False)
reasons_for_relocation_count_think_aloud

In [ ]:
# UHS descriptives for think-aloud participants
uhs_descriptives_think_aloud = df_think_aloud["UHS avg"].describe().round(2)
uhs_descriptives_think_aloud

## Correlations between UHS scores and listening task scores

In [ ]:
# The order of participants does not matter, as I use participants numbers (e.g. P1) in my thesis
# So I anonymize the data here
df_correlation["identifier"] = [f"P{i}" for i in range(1, len(df_correlation) + 1)]

In [ ]:
df_correlation["identifier"]

In [ ]:
df_correlation

In [ ]:
# Average personalness score of each recording for each participant
df_correlation["avg_personalness"] = df_correlation[["Rec 1 - Personalness", "Rec 2 - Personalness", 
                                                     "Rec 3 - Personalness", "Rec 4 - Personalness", 
                                                     "Rec 5 - Personalness"]].mean(axis=1)

# Average meaningfulness score of each recording for each participant
df_correlation["avg_meaningfulness"] = df_correlation[["Rec 1 - Meaningfulness", "Rec 2 - Meaningfulness", 
                                                       "Rec 3 - Meaningfulness", "Rec 4 - Meaningfulness", 
                                                       "Rec 5 - Meaningfulness"]].mean(axis=1)

# Average supportiveness score of each recording for each participant
df_correlation["avg_supportiveness"] = df_correlation[["Rec 1 - Supportiveness", "Rec 2 - Supportiveness", 
                                                       "Rec 3 - Supportiveness", "Rec 4 - Supportiveness", 
                                                       "Rec 5 - Supportiveness"]].mean(axis=1)

# Average personalness score of the three home country recordings for each participant
df_correlation["avg_home_personalness"] = df_correlation[["Rec 3 - Personalness", "Rec 4 - Personalness", "Rec 5 - Personalness"]].mean(axis=1)

# Average meaningfulness score of the three home country recordings for each participant
df_correlation["avg_home_meaningfulness"] = df_correlation[["Rec 3 - Meaningfulness", "Rec 4 - Meaningfulness", "Rec 5 - Meaningfulness"]].mean(axis=1)

# Average supportiveness score of the three home country recordings for each participant
df_correlation["avg_home_supportiveness"] = df_correlation[["Rec 3 - Supportiveness", "Rec 4 - Supportiveness", "Rec 5 - Supportiveness"]].mean(axis=1)

In [ ]:
# Correlation columns
potential_correlation_columns = ["avg_personalness", "avg_meaningfulness", "avg_supportiveness",
                                 "avg_home_personalness", "avg_home_meaningfulness", "avg_home_supportiveness",
                                 "Rec 3 - Personalness", "Rec 3 - Supportiveness", "Rec 4 - Meaningfulness",
                                 "Rec 5 - Meaningfulness"]

# Empty results list to put the correlation results in
results = []

# For each potential correlation column, take a subset per column and calculate the rho values and p-values of each variable
for column in potential_correlation_columns:
    subset = df_correlation[["UHS avg", column]].dropna()
    rho, p = spearmanr(subset["UHS avg"], subset[column])
    # add rho value and p-value to results
    results.append({"Variable is": column, "Rho": round(rho, 3), "p-value": round(p, 3)})

correlation_results = pd.DataFrame(results)
correlation_results

In [ ]:
# Create absolute rho value column
correlation_results["abs rho"] = correlation_results["Rho"].abs()

# Show correlations, from low to high
correlation_results = correlation_results.sort_values("abs rho", ascending=False)
correlation_results

In [ ]:
correlation_results.to_csv("UHS_correlation_results.csv", index=False)